# [INSTRUCTOR VERSION] BANA · Module 1 · Session 3
## Functions as Tools: lambda, map, filter, sorted, reduce and comprehensions

**Business case (continued):** *Namma Brew* café, Indiranagar. Today the owner asks:
*"Raise all prices by 10%. Show me only delivery orders. Rank orders by value. What did we earn in total?"*

Each question is answered with **one idea**: apply a function to many values, keep only what passes a test, rank by a rule, or combine everything into one number.

By the end of this session you will be able to:
1. Treat a function as a **value** you can pass to another function
2. Write short **lambda** functions
3. Transform and select data with **map**, **filter** and **comprehensions**
4. Rank with **sorted / max / min** using a `key`
5. Combine values with **reduce**
6. Explain why **pure functions** (no side effects) make analysis safer
7. Use the same ideas in **pandas** (`.map`, `.apply`)

**Rhythm:** Hook → Do → Debrief · **AI-use (AMBER):** syntax lookup allowed; reasoning and interpretation must be yours.

> **FACILITATOR NOTE**
>
> **Duration:** 60 minutes. **Timing:** Hook 0–5 · Functions as values 5–8 · lambda 8–12 · map/filter 12–20 · comprehensions 20–25 · sorted/key 25–29 · reduce 29–32 · pure functions 32–35 · pandas bridge 35–38 · Exercises F1–F5 38–55 · Exit ticket 55–60. F6–F8 go home.
> **Why this matters for business students:** map → filter → reduce is the mental model behind large-scale data processing (MapReduce; Dean & Ghemawat, 2008) and Apache Spark, the engine underneath Databricks (Zaharia et al., 2016). Say this in the Hook: "You are learning, on 8 rows, the pattern Databricks runs on billions."
> **Method:** PRIMM — students write a prediction in each `# PREDICT` comment before running (Sentance, Waite & Kallia, 2019).

---
## 0 · Today's data (run this first)

In [ ]:
menu = {"Filter Coffee": 40, "Cappuccino": 120, "Masala Chai": 30, "Bun Maska": 50, "Cold Brew": 150}

orders = [
    {"id": 101, "item": "Cappuccino",    "qty": 2,  "channel": "dine-in"},
    {"id": 102, "item": "Filter Coffee", "qty": 5,  "channel": "delivery"},
    {"id": 103, "item": "Bun Maska",     "qty": 1,  "channel": "dine-in"},
    {"id": 104, "item": "Cold Brew",     "qty": 3,  "channel": "delivery"},
    {"id": 105, "item": "Masala Chai",   "qty": 10, "channel": "delivery"},
    {"id": 106, "item": "Cappuccino",    "qty": 12, "channel": "dine-in"},
    {"id": 107, "item": "Filter Coffee", "qty": 2,  "channel": "takeaway"},
    {"id": 108, "item": "Bun Maska",     "qty": 6,  "channel": "takeaway"},
]

days       = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
daily_cups = [148, 132, 160, 95, 170, 210, 198]
print(f"{len(orders)} orders, {len(menu)} menu items loaded")

---
## 1 · HOOK — Where did my numbers go? (5 min)
The owner wants every price raised by 10%. A colleague wrote this. **Predict what prints**, then run.

In [ ]:
prices = [40, 120, 30, 50, 150]
new_prices = map(lambda p: round(p * 1.10), prices)

# PREDICT: ______________________
print(new_prices)

In [ ]:
# PREDICT: what do these two lines print?
print(list(new_prices))
print(list(new_prices))

**What happened?**
1. `map` does not compute anything immediately. It returns a **recipe** (an *iterator*) that produces values only when asked. `list()` asks for them.
2. An iterator can be read **only once**, like a till roll that has already been printed. The second `list()` finds nothing left.

**Rule:** if you need the results more than once, store them: `new_prices = list(map(...))`.

> **FACILITATOR NOTE**
>
> **Verified outputs:** `print(new_prices)` → `<map object at 0x…>`; first `list()` → `[44, 132, 33, 55, 165]`; second → `[]`.
> Lazy evaluation is the same reason Spark DataFrame transformations do not run until an *action* such as `count()` or `display()` is called — a useful bridge to Databricks later (Zaharia et al., 2016).

---
## 2 · Functions are values (3 min)
A function name without `()` is the function itself — you can store it or hand it to another function.

In [ ]:
def add_gst(amount):
    return round(amount * 1.05, 2)

tax_rule = add_gst          # no brackets: we store the function, we do not run it
print(tax_rule(100))        # PREDICT
print(add_gst)              # PREDICT: a number, or something else?

In [ ]:
def apply_to_all(func, values):
    """Run func on every value and return a new list."""
    results = []
    for v in values:
        results.append(func(v))
    return results

print(apply_to_all(add_gst, [100, 460, 2080]))

`apply_to_all` is a **higher-order function**: it takes another function as input. Python's built-in `map` does exactly this job.

---
## 3 · lambda — a function without a name (4 min)

```
lambda  input :  expression_to_return
```
Use a lambda for a **short, one-off rule** you pass straight into another function. If you need it twice, or it needs more than one line, write a normal `def`.

In [ ]:
def ten_percent_more(p):
    return round(p * 1.10)

same_thing = lambda p: round(p * 1.10)

print(ten_percent_more(120), same_thing(120))

In [ ]:
# Lambdas can take several inputs
line_total = lambda price, qty: price * qty
print(line_total(120, 3))     # PREDICT

# ...and can contain a one-line if/else (a "conditional expression")
size_tag = lambda qty: "bulk" if qty >= 10 else "regular"
print(size_tag(12), size_tag(2))

> **FACILITATOR NOTE**
>
> **Verified:** `add_gst(100)` → `105.0`; `print(add_gst)` → `<function add_gst at 0x…>`; `apply_to_all` → `[105.0, 483.0, 2184.0]`; `line_total(120,3)` → `360`; tags → `bulk regular`.
> **Style point:** PEP 8 discourages assigning a lambda to a name (`same_thing = lambda …`) — we do it here only to show equivalence with `def`. The name *lambda* comes from Church's lambda calculus (Church, 1936); students need only the one-line intuition.

---
## 4 · map and filter (8 min)

| Tool | Question it answers | Output length |
|---|---|---|
| `map(func, data)` | "Transform **every** item" | same as input |
| `filter(test, data)` | "Keep only items that **pass** the test" | same or shorter |

In [ ]:
# map: revenue of every order
order_revenue = list(map(lambda o: menu[o["item"]] * o["qty"], orders))
print(order_revenue)

In [ ]:
# filter: only delivery orders
delivery = list(filter(lambda o: o["channel"] == "delivery", orders))
print([o["id"] for o in delivery])     # PREDICT: which ids?

In [ ]:
# map with TWO lists: cups above/below target of 150, day by day
gap = list(map(lambda d, c: f"{d}: {c - 150:+d}", days, daily_cups))
print(gap)

### ⚠️ Trap: filter keeps anything "truthy"
`filter` keeps an item when the test returns something Python treats as **True**. Numbers other than 0 count as True.
A colleague wanted the days that **missed** 150 cups. **Predict**, then run.

In [ ]:
missed = list(filter(lambda c: c - 150, daily_cups))
print(missed)      # PREDICT: is this the list of days below 150?

In [ ]:
# Correct: the test must return a real True/False
missed = list(filter(lambda c: c < 150, daily_cups))
print(missed)

> **FACILITATOR NOTE**
>
> **Verified:** order revenue `[240, 200, 50, 450, 300, 1440, 80, 300]`; delivery ids `[102, 104, 105]`; gap `['Mon: -2', 'Tue: -18', 'Wed: +10', 'Thu: -55', 'Fri: +20', 'Sat: +60', 'Sun: +48']`.
> **Truthiness trap:** the buggy filter returns all seven values (no day is exactly 150, so `c - 150` is never 0). The silent output looks plausible — a deliberate "plausible-but-wrong" trap. Correct → `[148, 132, 95]`.

---
## 5 · Comprehensions — the Pythonic shortcut (5 min)

A **list comprehension** does map and filter in one readable line:

```
[ what_to_keep   for item in data   if test ]
     (map part)      (loop part)    (filter part)
```

In [ ]:
# map + filter together: revenue of delivery orders only
delivery_revenue = [menu[o["item"]] * o["qty"] for o in orders if o["channel"] == "delivery"]
print(delivery_revenue)

In [ ]:
# Dictionary comprehension: a new menu with 10% higher prices
new_menu = {item: round(price * 1.10) for item, price in menu.items()}
print(new_menu)
print(menu)        # PREDICT: did the original menu change?

In [ ]:
# Same result, three styles. Which reads most clearly to a colleague?
style_loop = []
for c in daily_cups:
    if c >= 150:
        style_loop.append(c)

style_filter = list(filter(lambda c: c >= 150, daily_cups))
style_comp   = [c for c in daily_cups if c >= 150]

print(style_loop == style_filter == style_comp, style_comp)

**Guidance:** most Python programmers prefer comprehensions for simple map/filter jobs because they read like English. `map` and `filter` are still worth knowing — you will meet them in pandas, Spark and other people's code.

> **FACILITATOR NOTE**
>
> **Verified:** delivery revenue `[200, 450, 300]`; new menu `{'Filter Coffee': 44, 'Cappuccino': 132, 'Masala Chai': 33, 'Bun Maska': 55, 'Cold Brew': 165}`; original unchanged; three styles → `True [160, 170, 210, 198]`.
> **Discussion prompt:** "Readable code is a control, not a nicety." The cost of an unreadable formula is that nobody catches its errors — link back to Session 2's spreadsheet stories.

---
## 6 · Ranking with sorted, max and min using `key` (4 min)
`key=` takes a function that says **what to rank by**.

In [ ]:
# Rank menu items by price, highest first
ranked = sorted(menu.items(), key=lambda pair: pair[1], reverse=True)
print(ranked)

In [ ]:
# Biggest order by revenue — max returns the whole ORDER, not just the number
biggest = max(orders, key=lambda o: menu[o["item"]] * o["qty"])
print(biggest)

# Best day: zip pairs days with cups, then rank by the cups
print(max(zip(days, daily_cups), key=lambda pair: pair[1]))

### ⚠️ Trap: `sorted()` vs `.sort()`

In [ ]:
cups_copy = daily_cups.copy()
result = cups_copy.sort()
print(result)       # PREDICT
print(cups_copy)

`sorted(data)` **returns a new list** and leaves the original alone. `list.sort()` **changes the list in place** and returns `None`.

> **FACILITATOR NOTE**
>
> **Verified:** ranked → `[('Cold Brew', 150), ('Cappuccino', 120), ('Bun Maska', 50), ('Filter Coffee', 40), ('Masala Chai', 30)]`; biggest → order 106 (`Cappuccino ×12`, ₹1,440); best day `('Sat', 210)`; `.sort()` returns `None`, list becomes `[95, 132, 148, 160, 170, 198, 210]`.
> This trap mirrors Session 2's `print` vs `return`: something *happened*, but nothing was *handed back*.

---
## 7 · reduce — fold many values into one (3 min)
`reduce` walks through a list, carrying a running result — the accumulator pattern from Session 2, as one call.

In [ ]:
from functools import reduce

total = reduce(lambda running, r: running + r, order_revenue, 0)
print(total)                      # PREDICT
print(sum(order_revenue))         # for simple totals, sum() is clearer

In [ ]:
# reduce shines when the combining rule is NOT just +
# Example: find the order with the largest quantity
largest_qty = reduce(lambda a, b: a if a["qty"] >= b["qty"] else b, orders)
print(largest_qty["id"], largest_qty["qty"])

In [ ]:
# ⚠️ Trap: what if there are no orders today? PREDICT
no_orders = []
print(reduce(lambda a, b: a + b, no_orders))

In [ ]:
# Fix: always give reduce a starting value
print(reduce(lambda a, b: a + b, no_orders, 0))

> **FACILITATOR NOTE**
>
> **Verified:** total `3060` both ways; largest qty → order `106, 12`; empty reduce → `TypeError: reduce() of empty iterable with no initial value`; with initial → `0`.
> **History point:** Python's creator moved `reduce` out of the built-ins into `functools` in Python 3 precisely because explicit loops or `sum()` are usually clearer (van Rossum, 2005). Teach it for reading others' code and for the MapReduce idea, not as a default.

---
## 8 · Pure functions — why "no side effects" matters (3 min)
A **pure** function only looks at its inputs and only gives back a result. It does not quietly change anything else.

In [ ]:
menu_test = {"Filter Coffee": 40, "Cappuccino": 120}

def festive_menu_impure(m):
    for item in m:
        m[item] = round(m[item] * 0.9)   # changes the ORIGINAL dictionary
    return m

festive = festive_menu_impure(menu_test)
print("festive  :", festive)
print("menu_test:", menu_test)   # PREDICT: did the regular menu survive?

In [ ]:
menu_test = {"Filter Coffee": 40, "Cappuccino": 120}

def festive_menu_pure(m):
    return {item: round(price * 0.9) for item, price in m.items()}   # builds a NEW dictionary

festive = festive_menu_pure(menu_test)
print("festive  :", festive)
print("menu_test:", menu_test)

**Business translation:** an impure function is like an intern who "prepares a discount sheet" by overwriting the master price list. Pure functions are easier to test, to rerun and to trust.

> **FACILITATOR NOTE**
>
> **Verified:** impure → both print `{'Filter Coffee': 36, 'Cappuccino': 108}`; pure → festive is discounted and `menu_test` stays `{'Filter Coffee': 40, 'Cappuccino': 120}`.
> Programs built from small, pure, composable functions are easier to reason about and reuse (Hughes, 1989; Backus, 1978) — the academic case for this style.

---
## 9 · Bridge to pandas (3 min)
The same ideas appear on DataFrame columns.

In [ ]:
import pandas as pd

df = pd.DataFrame(orders)
df["price"]   = df["item"].map(menu)                                  # map with a dictionary
df["revenue"] = df.apply(lambda row: row["price"] * row["qty"], axis=1)  # a function on each row
df["size"]    = df["qty"].map(lambda q: "bulk" if q >= 10 else "regular")
df

In [ ]:
# Faster and clearer for arithmetic: whole-column ("vectorised") operations
df["revenue_v"] = df["price"] * df["qty"]
print((df["revenue"] == df["revenue_v"]).all())
print(df.groupby("channel")["revenue"].sum())

**Rule of thumb:** use plain column arithmetic when you can; use `.map` / `.apply` with a lambda when the rule needs logic (if/else, look-ups).

> **FACILITATOR NOTE**
>
> **Verified:** revenue column matches `order_revenue`; vectorised check → `True`; channel totals → delivery `950`, dine-in `1730`, takeaway `380`.
> In Databricks, `df` (the last line of a cell) renders as a table; `display(df)` gives the interactive grid.

---
## 10 · Exercises
Pairs: driver types, navigator questions; swap each exercise. **F1–F5 in class · F6–F8 at home.**

### F1 · Predict the output (write all predictions first)

| # | Code | Prediction | Actual |
|---|---|---|---|
| a | `list(map(lambda x: x * 2, [1, 2, 3]))` | | |
| b | `list(filter(lambda x: x > 2, [1, 2, 3, 4]))` | | |
| c | `(lambda a, b: a - b)(10, 3)` | | |
| d | `sorted(["Tea", "coffee", "Bun"])` | | |
| e | `sorted(["Tea", "coffee", "Bun"], key=str.lower)` | | |
| f | `[x for x in range(6) if x % 2 == 0]` | | |
| g | `list(filter(None, [0, 1, "", "chai", None, 5]))` | | |
| h | `max(["Bun", "Cold Brew", "Tea"], key=len)` | | |
| i | `{k: v for k, v in [("a", 1), ("b", 2)]}` | | |
| j | `reduce(lambda a, b: a * b, [1, 2, 3, 4])` | | |

In [ ]:
# Check your predictions here
print(list(map(lambda x: x * 2, [1, 2, 3])))

In [ ]:
# ✅ SOLUTION
checks = ['list(map(lambda x: x * 2, [1, 2, 3]))', 'list(filter(lambda x: x > 2, [1, 2, 3, 4]))',
          '(lambda a, b: a - b)(10, 3)', 'sorted(["Tea", "coffee", "Bun"])',
          'sorted(["Tea", "coffee", "Bun"], key=str.lower)', '[x for x in range(6) if x % 2 == 0]',
          'list(filter(None, [0, 1, "", "chai", None, 5]))', 'max(["Bun", "Cold Brew", "Tea"], key=len)',
          '{k: v for k, v in [("a", 1), ("b", 2)]}', 'reduce(lambda a, b: a * b, [1, 2, 3, 4])']
for c in checks:
    print(f"{c:<50} -> {eval(c)!r}")
# Traps: (d) capitals sort before lowercase; (g) filter(None, …) drops every "falsy" value.

### F2 · Rewrite the loop
Each loop below works. Rewrite it **twice**: once with `map`/`filter`, once as a comprehension. Check all three give the same answer.

In [ ]:
# F2a — prices in paise (₹1 = 100 paise)
paise = []
for p in menu.values():
    paise.append(p * 100)
print(paise)

# YOUR map version:

# YOUR comprehension version:

In [ ]:
# F2b — names of items priced under ₹60
cheap = []
for item, price in menu.items():
    if price < 60:
        cheap.append(item)
print(cheap)

# YOUR filter (+ map) version:

# YOUR comprehension version:

In [ ]:
# F2c — order ids with quantity of 5 or more
big_ids = []
for o in orders:
    if o["qty"] >= 5:
        big_ids.append(o["id"])
print(big_ids)

# YOUR comprehension version:

In [ ]:
# ✅ SOLUTION
# F2a
print(list(map(lambda p: p * 100, menu.values())), [p * 100 for p in menu.values()])
# F2b
cheap_f = list(map(lambda pair: pair[0], filter(lambda pair: pair[1] < 60, menu.items())))
cheap_c = [item for item, price in menu.items() if price < 60]
print(cheap_f, cheap_c)
# F2c
print([o["id"] for o in orders if o["qty"] >= 5])
# Debrief: F2b shows nested map(filter(...)) gets hard to read — the comprehension wins.

### F3 · Fix the bugs
Each cell has **one** bug. Some crash; some are **silent**. Run, name the bug, fix it, and add a one-line comment.

In [ ]:
# F3a — show the new prices
raised = map(lambda p: p + 10, menu.values())
print("New prices:", raised)

In [ ]:
# F3b — count delivery orders, then list them
delivery_iter = filter(lambda o: o["channel"] == "delivery", orders)
print("How many:", len(list(delivery_iter)))
print("Which   :", [o["id"] for o in delivery_iter])

In [ ]:
# F3c — rank daily cups, highest first
ranked_cups = daily_cups.copy().sort(reverse=True)
print(ranked_cups[0])

In [ ]:
# F3d — keep only orders with quantity ABOVE 5
above_five = list(filter(lambda o: o["qty"] - 5, orders))
print([o["id"] for o in above_five])

In [ ]:
# F3e — total revenue from the new kiosk (no kiosk orders yet)
from functools import reduce
kiosk_orders = [o for o in orders if o["channel"] == "kiosk"]
print(reduce(lambda a, b: a + b, [menu[o["item"]] * o["qty"] for o in kiosk_orders]))

In [ ]:
# ✅ SOLUTION
# F3a SILENT: printing the map object, not its values -> wrap in list()
print("New prices:", list(map(lambda p: p + 10, menu.values())))

# F3b SILENT: iterator used up by the first list() -> store a list once
delivery_list = list(filter(lambda o: o["channel"] == "delivery", orders))
print("How many:", len(delivery_list), "| Which:", [o["id"] for o in delivery_list])

# F3c CRASH: .sort() returns None -> use sorted()
print(sorted(daily_cups, reverse=True)[0])

# F3d SILENT: truthiness — qty - 5 is "True" for every qty except 5 -> use a real comparison
print([o["id"] for o in filter(lambda o: o["qty"] > 5, orders)])

# F3e CRASH: reduce on an empty list with no start value -> give 0
print(reduce(lambda a, b: a + b, [menu[o["item"]] * o["qty"] for o in kiosk_orders], 0))

### F4 · Parsons puzzle — build a filter → map → reduce pipeline
Goal: **total revenue from delivery orders**. Put the lines in order in the empty cell (indentation is not needed — each is one line).

```
print(f"Delivery revenue: ₹{delivery_total}")
delivery_total = reduce(lambda a, b: a + b, delivery_values, 0)
from functools import reduce
delivery_orders = list(filter(lambda o: o["channel"] == "delivery", orders))
delivery_values = list(map(lambda o: menu[o["item"]] * o["qty"], delivery_orders))
```

In [ ]:
# ✅ SOLUTION
from functools import reduce
delivery_orders = list(filter(lambda o: o["channel"] == "delivery", orders))
delivery_values = list(map(lambda o: menu[o["item"]] * o["qty"], delivery_orders))
delivery_total = reduce(lambda a, b: a + b, delivery_values, 0)
print(f"Delivery revenue: ₹{delivery_total}")

### F5 · Write it — the owner's ranking report
Using `sorted`/`max`/`min` with `key=` and lambdas, print:
1. The **3 most expensive** menu items with prices
2. The order **id** with the **smallest** revenue
3. All orders sorted by **channel**, and within each channel by **revenue (highest first)** — print id, channel, revenue

*Hint for 3:* a key can return a pair, e.g. `key=lambda o: (o["channel"], -revenue)`. Python compares pairs left to right.

In [ ]:
# ✅ SOLUTION
top3 = sorted(menu.items(), key=lambda pair: pair[1], reverse=True)[:3]
print("Top 3:", top3)

rev = lambda o: menu[o["item"]] * o["qty"]
smallest = min(orders, key=rev)
print("Smallest order id:", smallest["id"], "revenue", rev(smallest))

for o in sorted(orders, key=lambda o: (o["channel"], -rev(o))):
    print(o["id"], o["channel"], rev(o))

### F6 · (Home) Functions that make functions
Write `make_discount(rate)` that **returns a function**. The returned function takes a price and gives back the discounted price, rounded to a whole rupee.
Then create `student_offer = make_discount(0.10)` and `staff_offer = make_discount(0.25)` and use `map` to apply each to the menu prices. Run the test cell.

In [ ]:
# ✅ SOLUTION
def make_discount(rate):
    return lambda price: round(price * (1 - rate))

In [ ]:
# Test cell — do not edit
student_offer = make_discount(0.10)
staff_offer   = make_discount(0.25)
tests = [(student_offer(120), 108), (staff_offer(120), 90), (student_offer(40), 36),
         (list(map(staff_offer, [40, 120, 30])), [30, 90, 22])]
for got, expected in tests:
    print("✅" if got == expected else "❌", got, "expected", expected)

💬 *Explain in one sentence:* how does `student_offer` still "remember" 10% after `make_discount` has finished running?

⚠️ Look at the last test: `0.75 × 30 = 22.5` became `22`, not `23`. Python rounds exact halves to the nearest **even** number ("banker's rounding"). Should a café bill use that rule? Write one line.

### F7 · (Home) pandas — tag and summarise
Using the DataFrame `df` from Section 9:
1. Add a column `peak` that is `"high value"` if revenue ≥ 400, else `"standard"` — use `.map` or `.apply` with a lambda
2. Add a column `item_upper` with the item name in capitals using `.map(str.upper)` (a function passed **without** brackets)
3. Print total revenue per `peak` group
4. Write **two sentences** to the owner: what share of revenue comes from high-value orders, and what she might do about it

In [ ]:
# ✅ SOLUTION
df["peak"] = df["revenue"].map(lambda r: "high value" if r >= 400 else "standard")
df["item_upper"] = df["item"].map(str.upper)
summary = df.groupby("peak")["revenue"].sum()
print(summary)
print(f"High-value share: {summary['high value'] / summary.sum():.1%}")
# Interpretation is the student's own (AMBER) — check that it uses the share, not just the totals.

### F8 · (Stretch) The lambda-in-a-loop trap
The owner wants three quick multipliers for 1×, 2× and 3× portion sizes. **Predict** the output, run it, then explain and fix.

In [ ]:
multipliers = [lambda price: price * n for n in [1, 2, 3]]
print([m(100) for m in multipliers])     # PREDICT

In [ ]:
# ✅ SOLUTION
# Every lambda looks up n when it RUNS, not when it was created — by then n is 3.
# Fix 1: freeze the current n as a default argument
multipliers = [lambda price, n=n: price * n for n in [1, 2, 3]]
print([m(100) for m in multipliers])
# Fix 2: use the function factory from F6
def make_multiplier(n):
    return lambda price: price * n
print([make_multiplier(n)(100) for n in [1, 2, 3]])

> **FACILITATOR NOTE**
>
> **Exercise design:** F1 prediction → F2 translation between styles → F3 debugging → F4 Parsons → F5–F8 writing, in rising difficulty. Parsons problems are an efficient bridge to writing (Ericson, Margulieux & Rick, 2017).
> **Verified answers:** F1 → `[2,4,6]`, `[3,4]`, `7`, `['Bun','Tea','coffee']`, `['Bun','coffee','Tea']`, `[0,2,4]`, `[1,'chai',5]`, `'Cold Brew'`, `{'a':1,'b':2}`, `24`. F2a `[4000,12000,3000,5000,15000]`; F2b `['Filter Coffee','Masala Chai','Bun Maska']`; F2c `[102,105,106,108]`. F3 originals: (a) `<map object>`, (b) `3` then `[]`, (c) `TypeError: 'NoneType' object is not subscriptable`, (d) ids `[101,103,104,105,106,107,108]` (only order 102, qty exactly 5, is dropped — even qty 1 and 2 pass) → fixed `[105,106,108]`; (e) `TypeError` → fixed `0`. F4 → `₹950`. F5 top 3 Cold Brew/Cappuccino/Bun Maska; smallest order `103` (₹50). F6 all ✅ (note `22` from banker's rounding). F7 high value ₹1,890 of ₹3,060 = `61.8%`. F8 → `[300,300,300]`, fixed `[100,200,300]`.
> **Viva prompts:** "Why does F3b print an empty list?" · "When would you choose a comprehension over map?" · "Is `festive_menu_impure` ever acceptable?"

---
## 11 · DEBRIEF — Exit ticket (5 min)
1. In plain English, match each tool to a business question: `map`, `filter`, `sorted(key=)`, `reduce`.
2. Which **silent** trap from today (lazy `map`, used-up iterator, truthiness, `.sort()`, impure function) is most likely to slip into a real report? Why?
3. Muddiest point?

---
## 12 · Keep going — resources

| Resource | Use it for |
|---|---|
| Python docs: *Functional Programming HOWTO* — docs.python.org/3/howto/functional.html | Iterators, map/filter, lambda, functools — the official explanation |
| Python tutorial §4.8 (lambda) and §5.1.3 (list comprehensions) — docs.python.org/3/tutorial | Short, authoritative syntax reference |
| McKinney, *Python for Data Analysis* 3e, Ch. 3.2 (functions) and Ch. 3.1 (comprehensions) — wesmckinney.com/book | Your prescribed text, with data-analysis examples |
| Kaggle Learn: Python — lessons on functions and loops/comprehensions | Quick auto-checked practice |
| Python Tutor — pythontutor.com | Watch iterators get used up and closures remember values |
| pandas user guide: *Applying functions* — pandas.pydata.org/docs | `.map`, `.apply`, and when to prefer vectorised code |

**Challenge for the week:** take any loop you wrote in Session 2 and rewrite it as a comprehension. Then decide which version a colleague would understand faster — and why.

---
## Facilitator references

- Backus, J. (1978). Can programming be liberated from the von Neumann style? A functional style and its algebra of programs. *Communications of the ACM, 21*(8), 613–641.
- Church, A. (1936). An unsolvable problem of elementary number theory. *American Journal of Mathematics, 58*(2), 345–363.
- Dean, J., & Ghemawat, S. (2008). MapReduce: Simplified data processing on large clusters. *Communications of the ACM, 51*(1), 107–113.
- Ericson, B. J., Margulieux, L. E., & Rick, J. (2017). Solving Parsons problems versus fixing and writing code. *Proceedings of Koli Calling '17*, 20–29. ACM.
- Hughes, J. (1989). Why functional programming matters. *The Computer Journal, 32*(2), 98–107.
- McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly.
- Python Software Foundation. *Functional Programming HOWTO*; *PEP 8 – Style Guide for Python Code*. docs.python.org / peps.python.org.
- Sentance, S., Waite, J., & Kallia, M. (2019). Teaching computer programming with PRIMM: A sociocultural perspective. *Computer Science Education, 29*(2–3), 136–176.
- van Rossum, G. (2005). The fate of reduce() in Python 3000. *Artima Developer* (blog of Python's creator).
- Zaharia, M., Xin, R. S., Wendell, P., et al. (2016). Apache Spark: A unified engine for big data processing. *Communications of the ACM, 59*(11), 56–65.